In [78]:
import optuna

study_name = ""  # replace with your actual study name
# dataset = "Simulations_indep_traincontrol"
dataset = "NCT00113763"
n_trials = 150
# optuna_version_name = "ExMetrics2_bis_seedData{}_seedHPO{}".format(0, 10)
optuna_version_name = "ExMetrics2_seedHPO{}".format(10)
n_samples = 600
n_features_bytype = 6
treatment_effect = 0.
name_config = "simu_N{}_nfeat{}_t{}".format(n_samples, n_features_bytype, int(treatment_effect))
generator_name = "HI-VAE_weibull" # "HI-VAE_weibull" # "HI-VAE_piecewise" 
# study_name_cluster = "/home/pchassat/survgen-clinical-trials/dataset/{}/optuna_results/optuna_study_{}_ntrials{}_{}_{}".format(dataset, name_config, n_trials, optuna_version_name, generator_name)
# db_file = "/Users/pchassat/Documents/survgen-clinical-trials/dataset/{}/optuna_results/optuna_study_{}_ntrials{}_{}_{}.db".format(dataset, name_config, n_trials, optuna_version_name, generator_name)
study_name_cluster = "/home/pchassat/survgen-clinical-trials/dataset/{}/optuna_results/optuna_study_traincontrol_{}_ntrials{}_{}_{}".format(dataset, dataset, n_trials, optuna_version_name, generator_name)
db_file = "/Users/pchassat/Documents/survgen-clinical-trials/dataset/{}/optuna_results/optuna_study_traincontrol_{}_ntrials{}_{}_{}.db".format(dataset, dataset, n_trials, optuna_version_name, generator_name)
storage = f"sqlite:///{db_file}"
study = optuna.load_study(study_name=study_name_cluster, storage=storage)
names_objs = ["Survival curves dist","Identifiability score"]

In [79]:
from optuna.visualization import plot_parallel_coordinate, plot_slice, plot_param_importances

for i in range(len(names_objs)):
    plot_parallel_coordinate(study, target=lambda t: t.values[i], target_name=names_objs[i]).show() # relationships between objectives and parameters

In [80]:
for i in range(len(names_objs)):
    plot_slice(study, target=lambda t: t.values[i], target_name=names_objs[i]).show() # individual parameter effects

In [81]:
for i in range(len(names_objs)):
    print(f"Parameter importances for {names_objs[i]}:")
    plot_param_importances(study, target=lambda t: t.values[i], target_name=names_objs[i]).show() 

Parameter importances for Survival curves dist:


Parameter importances for Identifiability score:


In [82]:
# Get all trials on the Pareto front
pareto_trials = study.best_trials  # these are Pareto optimal

for t in pareto_trials:
    print("Values:", t.values)
    print("Params:", t.params)
    print("------")

Values: [0.010715342697455636, 0.4673684210526316]
Params: {'lr': 0.0001, 'batch_size': 76, 'z_dim': 120, 'y_dim': 60, 's_dim': 110}
------
Values: [0.009578143930989954, 0.47157894736842104]
Params: {'lr': 0.002, 'batch_size': 57, 'z_dim': 190, 'y_dim': 130, 's_dim': 170}
------
Values: [0.01577017611341698, 0.43157894736842106]
Params: {'lr': 0.0002, 'batch_size': 95, 'z_dim': 130, 'y_dim': 110, 's_dim': 200}
------
Values: [0.011540388464634696, 0.4421052631578947]
Params: {'lr': 0.003, 'batch_size': 95, 'z_dim': 150, 'y_dim': 10, 's_dim': 90}
------
Values: [0.019191025209445635, 0.42526315789473684]
Params: {'lr': 0.001, 'batch_size': 32, 'z_dim': 80, 'y_dim': 160, 's_dim': 30}
------
Values: [0.25658586444163634, 0.02526315789473684]
Params: {'lr': 0.005, 'batch_size': 32, 'z_dim': 160, 'y_dim': 200, 's_dim': 190}
------
Values: [0.011375490569619307, 0.4652631578947368]
Params: {'lr': 0.001, 'batch_size': 285, 'z_dim': 170, 'y_dim': 10, 's_dim': 90}
------


In [83]:
import pandas as pd

# df_pareto = pd.DataFrame([
#     {**t.params, **{names_objs[i]: v for i, v in enumerate(t.values)}}
#     for t in study.best_trials
# ])
# print(df_pareto)

df_pareto = pd.DataFrame([
    {
        "trial_number": t.number,          
        **t.params,
        **{names_objs[i]: v for i, v in enumerate(t.values)}
    }
    for t in study.best_trials
])

df_pareto.head(10)

,trial_number,lr,batch_size,z_dim,y_dim,s_dim,Survival curves dist,Identifiability score
0,7,0.0001,76,120,60,110,0.010715,0.467368
1,22,0.0020,57,190,130,170,0.009578,0.471579
2,27,0.0002,95,130,110,200,0.015770,0.431579
3,31,0.0030,95,150,10,90,0.011540,0.442105
4,86,0.0010,32,80,160,30,0.019191,0.425263
5,128,0.0050,32,160,200,190,0.256586,0.025263
6,134,0.0010,285,170,10,90,0.011375,0.465263


In [84]:
from optuna.visualization import plot_optimization_history

for i in range(len(names_objs)):
    plot_optimization_history(study, target=lambda t: t.values[i], target_name=names_objs[i]).show() 

In [85]:
from optuna.trial import TrialState
completed = [t for t in study.trials if t.state == TrialState.COMPLETE]
print("Number of trials:", len(study.trials))
print("Number of completed trials:", len(completed))

Number of trials: 150
Number of completed trials: 146


In [86]:
from optuna.visualization import plot_pareto_front
plot_pareto_front(
    study,
    targets=lambda t: [t.values[0], t.values[1]],
    target_names=["Survival curves dist", "Identifiability score"],
    include_dominated_trials=True
)

In [67]:
selected_trial_id = 103
best_trial = study.trials[selected_trial_id]

print("Values (objectifs):", best_trial.values)
print("Params:", best_trial.params)
print("State:", best_trial.state)
print("Start:", best_trial.datetime_start)
print("End:", best_trial.datetime_complete)
print("Duration:", best_trial.duration)

Values (objectifs): [0.009476586391039486, 0.11666666666666667]
Params: {'lr': 0.003, 'batch_size': 32, 'z_dim': 150, 'y_dim': 100, 's_dim': 190, 'n_layers_surv_piecewise': 2, 'n_intervals': 20}
State: 1
Start: 2026-06-17 18:01:06.798847
End: 2026-06-17 18:03:33.486546
Duration: 0:02:26.687699


### Save best selected trial

In [68]:
import json
parent_path = "/Users/pchassat/Documents/survgen-clinical-trials"
best_params_file = parent_path + "/dataset/" + dataset + "/optuna_results/best_params_{}_ntrials{}_{}_{}.json".format(name_config, n_trials, optuna_version_name, generator_name)
# best_params_file = parent_path + "/dataset/" + dataset + "/optuna_results/best_params_traincontrol_{}_ntrials{}_{}_{}.json".format(dataset, n_trials, optuna_version_name, generator_name)
with open(best_params_file, "w") as f:
    json.dump(best_trial.params, f)